importamos las librerias necesarias para el desarrollo del modelo

In [ ]:
import tensorflow as tf
from tensorflow import keras

import matplotlib.pyplot as plt

Descaragamos la libreria que estaba guiaada en la actividad


In [ ]:
import os
import tarfile
from urllib.request import urlretrieve

# URL y archivo local
url = "http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar"
tar_path = "images.tar"
extract_dir = "ImageNetDogs"

# Descargar (si no existe)
if not os.path.exists(tar_path):
    print("Descargando archivo...")
    urlretrieve(url, tar_path)
    print("Descarga completada.")
else:
    print("El archivo ya existe localmente.")

# Inspeccionar contenido del .tar sin extraer todo
print("\nPrimeros 30 elementos dentro del TAR:")
with tarfile.open(tar_path, "r") as tar:
    members = tar.getmembers()
    for m in members[:30]:
        print(m.name)

print(f"\nTotal de elementos en el TAR: {len(members)}")

# Extraer (opcional, si quieres ver carpetas/archivos en disco)
if not os.path.exists(extract_dir):
    print("\nExtrayendo contenido...")
    with tarfile.open(tar_path, "r") as tar:
        tar.extractall(path=extract_dir)
    print("Extracción completada.")
else:
    print("\nEl contenido ya fue extraído previamente.")

# Mostrar estructura de primer nivel extraída
print("\nContenido de primer nivel en carpeta extraída:")
for item in os.listdir(extract_dir)[:30]:
    print(item)

Separamos la imegens del entrenamiento con el testeo haciendo la fusion del 80 % y el 20 %

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{extract_dir}/Images",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(32, 32),
    batch_size=32
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    f"{extract_dir}/Images",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(32, 32),
    batch_size=32
)

print("Lotes de entrenamiento:", tf.data.experimental.cardinality(train_ds).numpy())
print("Lotes de prueba:", tf.data.experimental.cardinality(test_ds).numpy())

Se muestra la primera imgen del banco de imagene de testeo del dataset

In [ ]:
# Cantidad total de imágenes en train y test
num_train_images = tf.data.experimental.cardinality(train_ds).numpy() * 32
num_test_images = tf.data.experimental.cardinality(test_ds).numpy() * 32
num_total_images = num_train_images + num_test_images

print(f"Cantidad aproximada de imágenes de entrenamiento: {num_train_images}")
print(f"Cantidad aproximada de imágenes de testeo: {num_test_images}")
print(f"Cantidad aproximada total de imágenes: {num_total_images}")

# Mostrar la primera imagen del conjunto de test usando matplotlib
for images, labels in test_ds.take(1):
    first_image = images[0].numpy().astype("uint8")
    first_label = labels[0].numpy()

plt.figure(figsize=(4, 4))
plt.imshow(first_image)
plt.title(f"Primera imagen de test (label: {first_label})")
plt.axis("off")
plt.show()

Se normalizan o svalores en 255 ya que son los colores que estan ya que las imagene se encuentra a color y damos sus variables

In [ ]:
import math

# Obtener los nombres de las clases
class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Número de clases: {num_classes}")

# Obtener una imagen por clase
class_images = {}
for images_batch, labels_batch in train_ds:
	for img, label in zip(images_batch, labels_batch):
		label_idx = label.numpy()
		if label_idx not in class_images:
			class_images[label_idx] = img.numpy().astype("uint8")
		if len(class_images) == num_classes:
			break
	if len(class_images) == num_classes:
		break

# Mostrar todas las clases con su imagen
cols = 10
rows = math.ceil(num_classes / cols)

plt.figure(figsize=(cols * 2.5, rows * 2.5))
for idx in range(num_classes):
	plt.subplot(rows, cols, idx + 1)
	if idx in class_images:
		plt.imshow(class_images[idx])
	plt.title(class_names[idx].split("-")[-1][:10], fontsize=6)
	plt.axis("off")

plt.suptitle(f"Las {num_classes} clases del dataset ImageNet Dogs", fontsize=14)
plt.tight_layout()
plt.show()

con esto se hac la normlizacion del modelo

In [ ]:

x_train = train_ds.map(lambda x, y: (x / 255.0, y))
x_test = test_ds.map(lambda x, y: (x / 255.0, y))

print("x_train:", x_train)
print("x_test:", x_test)

Aqui miramos si esta normalizado las imagenes que tenemos en el dataset

In [ ]:
for x_batch, y_batch in x_train.take(1):
    print("Forma del lote:", x_batch.shape)
    print("Etiquetas del lote:", y_batch[:10].numpy())
    print("Valor mínimo:", tf.reduce_min(x_batch).numpy())
    print("Valor máximo:", tf.reduce_max(x_batch).numpy())
    print("Primeros valores normalizados del primer píxel:", x_batch[0, 0, 0].numpy())

plt.figure(figsize=(4, 4))
plt.imshow(x_batch[0].numpy())
plt.title(f"Imagen normalizada (label: {y_batch[0].numpy()})")
plt.axis("off")
plt.show()

Aqui hacemos modelo convulucionales con esto en este caso es dos convoluciones de 64 y de 31 junto con sus maxpooling de 2x2 para la actividad

In [ ]:
modelo = keras.Sequential([
    keras.layers.Input(shape=(32, 32, 3)),
    keras.layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(),
    keras.layers.Flatten(),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dropout(0.4),
    keras.layers.Dense(len(train_ds.class_names), activation='softmax')
])


Aqui la generacion del entrenamiento del modelo t tambien se agrega 10 epocas 

In [ ]:
modelo.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2
)

history = modelo.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10,
    callbacks=[callback]
)


se muestra el hsitorial del modelo

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history.history["accuracy"], label="accuracy", marker="o")
plt.plot(history.history["val_accuracy"], label="val_accuracy", marker="o")

plt.title("Evolución de Accuracy y Val Accuracy")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

se muestra una imgen de wikipedia si hace la prediccion del modelo

In [ ]:
import numpy as np
import requests
from PIL import Image
from io import BytesIO

# Descargar imagen de Wikipedia (Golden Retriever)
img_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/b/bd/Golden_Retriever_Adorable.jpg/320px-Golden_Retriever_Adorable.jpg"
response = requests.get(img_url)
img = Image.open(BytesIO(response.content)).convert("RGB")
img_resized = img.resize((32, 32))

# Mostrar la imagen descargada
plt.figure(figsize=(4, 4))
plt.imshow(img_resized)
plt.title("Imagen de Wikipedia")
plt.axis("off")
plt.show()

# Preprocesar la imagen para el modelo
img_array = np.array(img_resized) / 255.0
img_array = np.expand_dims(img_array, axis=0)  # shape: (1, 224, 224, 3)

# Predecir con el modelo entrenado
prediccion = modelo.predict(img_array)
clase_predicha = np.argmax(prediccion)
confianza = np.max(prediccion) * 100

# Obtener nombre de la clase
class_names = train_ds.class_names
nombre_clase = class_names[clase_predicha]

print(f"Clase predicha (índice): {clase_predicha}")
print(f"Nombre de la clase: {nombre_clase}")
print(f"Confianza: {confianza:.2f}%")

Se tomas las imgenes del banco de teseto junto con su resultado predictivo

In [ ]:
# Tomar las primeras 10 imágenes del conjunto de test y compararlas con las predicciones del modelo
for x_lote, y_lote in x_test.take(1):
	imgs_10 = x_lote[:10]
	labels_10 = y_lote[:10]

preds = modelo.predict(imgs_10, verbose=0)
pred_classes = tf.argmax(preds, axis=1).numpy()
true_classes = labels_10.numpy()
confianzas = tf.reduce_max(preds, axis=1).numpy()

class_names = train_ds.class_names

plt.figure(figsize=(20, 8))
for i in range(10):
	plt.subplot(2, 5, i + 1)
	plt.imshow(imgs_10[i].numpy())
	
	real = class_names[true_classes[i]]
	pred = class_names[pred_classes[i]]
	conf = confianzas[i] * 100
	ok = "✅" if pred_classes[i] == true_classes[i] else "❌"
	
	plt.title(f"{ok}\nReal: {real}\nPred: {pred}\n{conf:.1f}%", fontsize=9)
	plt.axis("off")

plt.tight_layout()
plt.show()

# Resumen en texto
aciertos = (pred_classes == true_classes).sum()
print(f"Aciertos en las primeras 10 imágenes: {aciertos}/10")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np

# Obtener todas las predicciones y etiquetas reales del conjunto de test
all_preds = []
all_labels = []

for x_lote, y_lote in x_test:
    preds = modelo.predict(x_lote, verbose=0)
    all_preds.extend(np.argmax(preds, axis=1))
    all_labels.extend(y_lote.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Calcular la matriz de confusión
cm = confusion_matrix(all_labels, all_preds)

# Mostrar la matriz de confusión
fig, ax = plt.subplots(figsize=(30, 30))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[c.split("-")[-1][:10] for c in class_names])
disp.plot(ax=ax, xticks_rotation=90, colorbar=False, cmap="Blues")

plt.title("Matriz de Confusión - ImageNet Dogs", fontsize=16)
plt.tight_layout()
plt.show()

# Resumen de accuracy global
accuracy = (all_preds == all_labels).sum() / len(all_labels)
print(f"Accuracy global en test: {accuracy * 100:.2f}%")


In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))
# Si da lista vacía, ve a Entorno de ejecución → Cambiar tipo → GPU T4